In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
import utils

# Suppress scientific notation
np.set_printoptions(suppress=True)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')


In [ ]:
DEBUG = True
datasets, seeds, percentages = utils.get_config(DEBUG)


In [ ]:
def compute_loss(pred, labels):
    return F.binary_cross_entropy(pred, labels)

def calculate_features(G_nx, edge_list):
    rai_gen = nx.resource_allocation_index(G_nx, edge_list)
    rai     = {(u, v): p for u, v, p in rai_gen}

    jc_gen  = nx.jaccard_coefficient(G_nx, edge_list)
    jc      = {(u, v): p for u, v, p in jc_gen}

    aa_gen  = nx.adamic_adar_index(G_nx, edge_list)
    aa      = {(u, v): p for u, v, p in aa_gen}

    pa_gen  = nx.preferential_attachment(G_nx, edge_list)
    pa      = {(u, v): p for u, v, p in pa_gen}

    shortest_paths = dict(nx.all_pairs_shortest_path_length(G_nx, cutoff=5))

    features = []
    for u, v in edge_list:
        jc_score  = jc.get((u, v), 0.0)
        aa_score  = aa.get((u, v), 0.0)
        rai_score = rai.get((u, v), 0.0)
        pa_score  = pa.get((u, v), 0.0)
        spl       = shortest_paths.get(u, {}).get(v, 6)
        num_paths = 1
        features.append([jc_score, aa_score, rai_score, pa_score, spl, num_paths])
    return np.array(features)


In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # Input shape will be (N, 1, 6, 1)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=(3,1))
        self.bn1   = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=(3,1))
        self.bn2   = nn.BatchNorm2d(32)
        # After conv1: height: 6-3+1=4; width=1
        # After conv2: height: 4-3+1=2; width=1
        # Flatten: 32 * 2 * 1 = 64
        self.fc   = nn.Linear(64, 1)

    def forward(self, x):
        # x: (N, 6)
        x = x.unsqueeze(1).unsqueeze(3)       # (N,1,6,1)
        x = F.relu(self.bn1(self.conv1(x)))   # (N,64,4,1)
        x = F.relu(self.bn2(self.conv2(x)))   # (N,32,2,1)
        x = x.view(x.size(0), -1)             # (N,64)
        x = torch.sigmoid(self.fc(x))         # (N,1)
        return x


In [ ]:
# Main processing loops
for seed in seeds:
    print(f'Processing seed {seed}')
    for ds in datasets:
        print(f'Analyzing {ds} dataset with seed {seed}')

        # Assuming data folder is local for this refactored notebook
        # You may need to adjust the path based on your data directory structure
        path = f'./data/w_removal_{ds}'
        if not os.path.exists(path):
            print(f"Data file not found at {path}. Please place 'w_removal_{ds}' in the './data/' folder.")
            continue
            
        data_total = pd.read_csv(path, sep=" ", header=None,
                                 names=["n1","n2","f1","f2","f3","f4","f5","f6","f7","l"])
        data_total.dropna(subset=['n1','n2'], inplace=True)

        # Map node IDs to contiguous indices
        nodes       = np.unique(data_total[['n1','n2']].values)
        node_index  = {node: idx for idx, node in enumerate(nodes)}
        num_nodes   = len(nodes)

        edges       = data_total[data_total['l'] == 1][['n1','n2']].values
        edges_mapped= np.array([[node_index[u], node_index[v]] for u, v in edges])

        # Build full graph
        G = nx.Graph()
        G.add_nodes_from(range(num_nodes))
        G.add_edges_from(edges_mapped)

        # All possible node pairs
        possible_edges = [(i, j) for i in range(num_nodes) for j in range(i+1, num_nodes)]
        total_edges    = len(possible_edges)

        # Shuffle once per seed
        np.random.seed(seed)
        np.random.shuffle(possible_edges)

        for perc in percentages:
            pct_int = int(perc * 100)
            print(f'  ▶ test/train split: {pct_int}% test')

            # Split edges
            test_size   = int(perc * total_edges)
            test_edges  = possible_edges[:test_size]
            train_edges = possible_edges[test_size:]

            # Create training graph
            G_train = G.copy()
            G_train.remove_edges_from(test_edges)

            # Compute features + labels
            print("    Calculating training features…")
            train_features = calculate_features(G_train, train_edges)
            train_labels   = np.array([1 if G.has_edge(u, v) else 0 for u, v in train_edges])

            print("    Calculating test features…")
            test_features = calculate_features(G_train, test_edges)
            test_labels   = np.array([1 if G.has_edge(u, v) else 0 for u, v in test_edges])

            # Standardize
            scaler      = StandardScaler()
            X_train_np  = scaler.fit_transform(train_features)
            X_test_np   = scaler.transform(test_features)

            # To tensors
            X_train = torch.tensor(X_train_np, dtype=torch.float32).to(device)
            y_train = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1).to(device)
            X_test  = torch.tensor(X_test_np, dtype=torch.float32).to(device)
            y_test  = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1).to(device)

            # Build model & optimizer
            model     = CNN().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

            # Training
            num_epochs  = 10 # lower for debug, but loop is short anyway
            if DEBUG:
                num_epochs = 2 # even shorter
                
            batch_size  = 8192
            num_batches = int(np.ceil(X_train.size(0) / batch_size))
            model.train()
            for epoch in range(num_epochs):
                permutation = torch.randperm(X_train.size(0)).to(device)
                epoch_loss  = 0.0
                for i in range(num_batches):
                    idx      = permutation[i*batch_size:(i+1)*batch_size]
                    batch_x  = X_train[idx]
                    batch_y  = y_train[idx]

                    optimizer.zero_grad()
                    outputs = model(batch_x)
                    loss    = compute_loss(outputs, batch_y)
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()

                if epoch % 1 == 0:
                    avg_loss = epoch_loss / max(1, num_batches)
                    print(f'    Epoch {epoch:3d}  Loss: {avg_loss:.4f}')

            # Evaluation
            model.eval()
            with torch.no_grad():
                outputs = model(X_test)
                preds   = (outputs >= 0.5).float()
                y_true  = y_test.cpu().numpy()
                y_scores= outputs.cpu().numpy()
                y_pred  = preds.cpu().numpy()

                auc = roc_auc_score(y_true, y_scores)
                acc = np.mean(y_pred == y_true)
                cm  = confusion_matrix(y_true, y_pred)
                ber = utils.balanced_error_rate(cm)
                print(f'    → AUC: {auc:.4f}  ACC: {acc:.4f}  BER: {ber:.4f}')

            # Prepare directories
            base_dir   = f'./results/CNN/{ds}'
            dirs = {
                'cm_png':  os.path.join(base_dir, 'cm_png',  str(seed), str(pct_int)),
                'cm_npy':  os.path.join(base_dir, 'cm_npy',  str(seed), str(pct_int)),
                'models':  os.path.join(base_dir, 'models',  str(seed), str(pct_int)),
                'csv':     os.path.join(base_dir, 'csv',     str(seed), str(pct_int)),
            }
            for path_dir in dirs.values():
                os.makedirs(path_dir, exist_ok=True)

            # Save confusion matrix plot
            plt.figure(figsize=(6,6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                        xticklabels=['Pred 0','Pred 1'], yticklabels=['Act 0','Act 1'])
            plt.title(f'CM {ds} seed{seed} {pct_int}%')
            png_path = os.path.join(dirs['cm_png'], f'cm_{ds}_seed{seed}_p{pct_int}.png')
            plt.savefig(png_path, dpi=200, bbox_inches='tight')
            plt.close()

            # Save numpy cm
            np.save(os.path.join(dirs['cm_npy'], f'cm_{ds}_seed{seed}_p{pct_int}.npy'), cm)

            # Save model
            torch.save(model.state_dict(),
                       os.path.join(dirs['models'], f'model_{ds}_seed{seed}_p{pct_int}.pth'))

            # Save results CSV
            res = [{
                'Dataset':    ds,
                'Seed':       seed,
                'Percentage': perc,
                'AUC':        auc,
                'ACC':        acc,
                'BER':        ber
            }]
            pd.DataFrame(res).to_csv(
                os.path.join(dirs['csv'], f'results_{ds}_seed{seed}_p{pct_int}.csv'),
                index=False
            )
